# Train biped_rl on Kaggle
1. Upload `kaggle_upload.zip` as a Kaggle Dataset, add it to this notebook (Add Input).
2. Settings: **Internet on**. Accelerator does not matter (physics runs on CPU); pick none or GPU, both give ~30 GB RAM.
3. *Save Version -> Save & Run All* to train unattended (max 12 h). Download `stand_b_out.zip` from Output when done.

In [ ]:
import glob, os, shutil
RUN, START_LEVEL, KP_SCALE, STEPS = "stand_b", 3, 3, 3_000_000   # edit here
src = os.path.dirname(glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)[0])
shutil.copytree(src, "/kaggle/working/biped_rl", dirs_exist_ok=True)
os.chdir("/kaggle/working/biped_rl")
print(src, "->", os.getcwd(), "| cpus:", os.cpu_count())

In [ ]:
!pip install -q "mujoco>=3.2" "gymnasium>=1.0" "stable-baselines3>=2.3" imageio-ffmpeg
!pip install -q -e . --no-deps

In [ ]:
# resume from the newest checkpoint (latest.zip is stale if the last run was killed)
import re
run = f"runs/{RUN}"
steps = max(int(re.findall(r"ckpt_(\d+)_steps", f)[0]) for f in glob.glob(f"{run}/ckpt_*_steps.zip"))
shutil.copy(f"{run}/ckpt_{steps}_steps.zip", f"{run}/latest.zip")
shutil.copy(f"{run}/ckpt_vecnormalize_{steps}_steps.pkl", f"{run}/vecnormalize.pkl")
print("resuming from", steps)

In [ ]:
N_ENVS = os.cpu_count()
!PYTHONUNBUFFERED=1 python scripts/train.py --name {RUN} --resume runs/{RUN}/latest.zip --start-level {START_LEVEL}     --kp-scale {KP_SCALE} --total-steps {STEPS} --n-envs {N_ENVS} --save-every 250000 2>&1 | tee -a runs/{RUN}/train.log | grep -E "level|speed |ep_len_mean|fps|total_timesteps|Error" 

In [ ]:
shutil.make_archive(f"/kaggle/working/{RUN}_out", "zip", "runs", RUN)
print("download", f"{RUN}_out.zip", "from the Output tab, unzip into runs/")